In [1]:
from huggingface_hub import hf_hub_download, snapshot_download
import os

os.makedirs("../model/marbert_base", exist_ok=True)

print("Downloading chỉ file safetensors (bỏ qua .bin)...")

# Download toàn bộ repo nhưng ignore file .bin
snapshot_download(
    repo_id="UBC-NLP/MARBERTv2",
    local_dir="../model/marbert_base",
    ignore_patterns=["*.bin", "*.msgpack", "flax_model*", "tf_model*", "rust_model*"],
)

print("Xong! Kiểm tra thư mục:")
for f in os.listdir("../model/marbert_base"):
    print(f"  {f}")

d:\StanceEval-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 11.31it/s]

Xong! Kiểm tra thư mục:
  .cache
  .gitattributes
  config.json
  README.md
  special_tokens_map.json
  tokenizer_config.json
  vocab.txt


In [2]:
import torch
from transformers import AutoModel, AutoTokenizer
from safetensors.torch import save_file

print("Repo không có safetensors, tự convert từ .bin...")

# Load unsafe (bypass check) — chỉ dùng 1 lần để convert
import transformers.modeling_utils as mu
original_check = mu.check_torch_load_is_safe
mu.check_torch_load_is_safe = lambda: None  # monkey-patch tạm thời

model = AutoModel.from_pretrained("UBC-NLP/MARBERTv2")
tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/MARBERTv2")

mu.check_torch_load_is_safe = original_check  # restore

# Save lại dạng safetensors
model.save_pretrained("../model/marbert_base", safe_serialization=True)
tokenizer.save_pretrained("../model/marbert_base")

print("Convert xong! File model.safetensors đã có trong ../model/marbert_base")

Repo không có safetensors, tự convert từ .bin...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5058.16it/s]
[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.94s/it]

Convert xong! File model.safetensors đã có trong ../model/marbert_base
